# Parameter Sensitivity Analysis - Kronos

This notebook performs a parameter sensitivity analysis on the Kronos model to evaluate the impact of different data parameters on model performance.

## Setup

In [ ]:
import os
from pathlib import Path

REPO_NAME = "ba"
REPO_URL = "https://github.com/bp571/ba.git"

if not Path(REPO_NAME).exists():
    print(f"Cloning repository: {REPO_URL}")
    !git clone {REPO_URL}
else:
    print(f"Repository already cloned: {REPO_NAME}")

os.chdir(REPO_NAME)
print(f"Current directory: {os.getcwd()}")

In [ ]:
!pip install -q torch transformers peft gluonts SALib PyYAML tqdm pandas numpy matplotlib seaborn

## Configuration

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

METHOD = 'sobol'  # 'sobol' or 'grid'
SEED = 42
BATCH_SIZE = 48
QUICK_MODE = True  # Set to False for full analysis
MAX_WINDOWS = 5 if QUICK_MODE else None
N_SAMPLES = 8 if QUICK_MODE else None

print(f"Method: {METHOD}")
print(f"Seed: {SEED}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Quick mode: {QUICK_MODE}")
if MAX_WINDOWS:
    print(f"Max windows: {MAX_WINDOWS}")
if N_SAMPLES:
    print(f"N samples: {N_SAMPLES}")

## Run Sensitivity Analysis

In [ ]:
import json
import time
import numpy as np
from SALib.sample import sobol as sobol_sample
import yaml
from tqdm import tqdm
import pandas as pd

from data.factory import DataFactory
from core.model_loader import load_kronos_predictor
from core.reproducibility import set_all_seeds
from experiments.runner import run_rolling_benchmark_multi_asset

def load_config(config_path="03_sensitivity_analysis/data_parameters/parameter_space.yaml"):
    with open(config_path) as f:
        return yaml.safe_load(f)

def generate_grid_samples(config):
    from itertools import product
    
    param_names = list(config['parameter_space'].keys())
    
    if 'univariate_values' in config:
        grid_values = {name: config['univariate_values'][name] for name in param_names}
    else:
        n_points = config['sampling'].get('grid_points', 5)
        grid_values = {}
        for name in param_names:
            pmin = config['parameter_space'][name]['min']
            pmax = config['parameter_space'][name]['max']
            grid_values[name] = np.linspace(pmin, pmax, n_points, dtype=int).tolist()
    
    param_configs = []
    for combo in product(*[grid_values[name] for name in param_names]):
        param_configs.append({name: value for name, value in zip(param_names, combo)})
    
    return param_configs

def generate_parameter_samples(config, method='sobol', n_override=None):
    if method == 'grid':
        return generate_grid_samples(config)
    
    param_names = list(config['parameter_space'].keys())
    
    problem = {
        'num_vars': len(param_names),
        'names': param_names,
        'bounds': [
            [config['parameter_space'][p]['min'] - 0.5, 
             config['parameter_space'][p]['max'] + 0.5]
            for p in param_names
        ]
    }
    
    n_samples = n_override if n_override else config['sampling']['n_samples']
    seed = config['sampling']['seed']
    
    samples = sobol_sample.sample(problem, n_samples, calc_second_order=False, seed=seed)
    
    Path("03_sensitivity_analysis/data_parameters/results/raw_sobol").mkdir(parents=True, exist_ok=True)
    np.save("03_sensitivity_analysis/data_parameters/results/raw_sobol/sobol_X.npy", samples)
    
    param_configs = []
    for sample in samples:
        param_configs.append({name: int(np.round(value)) for name, value in zip(param_names, sample)})
    
    return param_configs

def prepare_asset_data(config_path, seed):
    factory = DataFactory(config_path=config_path)
    tickers = factory.get_tickers()
    
    asset_data = {}
    for ticker in tickers:
        try:
            df = factory.load_or_download(ticker)
            if df.empty:
                continue
            
            test_start = pd.Timestamp('2021-01-01', tz='UTC')
            if isinstance(df.index, pd.DatetimeIndex):
                df = df[df.index >= test_start]
            elif 'datetime' in df.columns:
                df['datetime'] = pd.to_datetime(df['datetime'])
                df = df[df['datetime'] >= test_start]
            
            if df.empty:
                continue
            
            if 'datetime' not in df.columns:
                df = df.reset_index().rename(columns={df.index.name: 'datetime', 'date': 'datetime'})
            
            asset_data[ticker] = df
        except Exception as e:
            print(f"Error loading {ticker}: {e}")
            continue
    
    return asset_data

def run_experiment(params, experiment_id, asset_data, predictor, batch_size, output_dir, max_windows=None):
    context = params['context_steps']
    forecast = params['forecast_steps']
    stride = forecast
    
    min_data_length = min(len(df) for df in asset_data.values())
    max_steps = (min_data_length - context - forecast) // stride + 1
    
    if max_steps <= 0:
        return None
    
    if max_windows is not None:
        max_steps = min(max_steps, max_windows)
    
    run_params = {
        'context_steps': context,
        'forecast_steps': forecast,
        'stride_steps': stride,
        'steps': max_steps
    }
    
    try:
        results = run_rolling_benchmark_multi_asset(
            predictor=predictor,
            asset_data_dict=asset_data,
            params=run_params,
            batch_size=batch_size,
            verbose=False
        )
        
        output_file = output_dir / f"exp_{experiment_id:04d}.json"
        with open(output_file, 'w') as f:
            json.dump({
                'experiment_id': experiment_id,
                'parameters': {**params, 'stride_steps': stride},
                'max_steps': max_steps,
                'n_assets': len(results),
                'results': {
                    ticker: {
                        'metrics': result['metrics'],
                        'n_predictions': len(result['raw_values']['actual'])
                    }
                    for ticker, result in results.items()
                }
            }, f, indent=2)
        
        return results
    except Exception as e:
        print(f"Experiment {experiment_id} failed: {e}")
        return None

print("="*80)
print("PARAMETER SENSITIVITY ANALYSIS - KRONOS")
print("="*80)

config = load_config()
param_samples = generate_parameter_samples(config, method=METHOD, n_override=N_SAMPLES)

print(f"Generated {len(param_samples)} parameter configurations")

output_dir = Path(f"03_sensitivity_analysis/data_parameters/results/raw_{METHOD}")
output_dir.mkdir(parents=True, exist_ok=True)

asset_data = prepare_asset_data("03_sensitivity_analysis/data_parameters/assets_sensitivity.yaml", SEED)
print(f"Loaded {len(asset_data)} assets")

set_all_seeds(seed=SEED)
predictor = load_kronos_predictor()

print("Running experiments...")
start_time = time.time()

results = []
for i, params in enumerate(tqdm(param_samples, desc="Progress")):
    result = run_experiment(
        params=params,
        experiment_id=i,
        asset_data=asset_data,
        predictor=predictor,
        batch_size=BATCH_SIZE,
        output_dir=output_dir,
        max_windows=MAX_WINDOWS
    )
    results.append(result)

elapsed = time.time() - start_time

print("")
print("="*80)
print("COMPLETED")
print("="*80)
print(f"Total experiments: {len(param_samples)}")
print(f"Successful: {sum(1 for r in results if r is not None)}")
print(f"Time: {elapsed:.1f}s ({elapsed/len(param_samples):.1f}s per experiment)")
print(f"Results saved to: {output_dir}")

## Analyze Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from SALib.analyze import sobol as sobol_analyze

def load_results(results_dir):
    results_path = Path(results_dir)
    experiment_files = sorted(results_path.glob("exp_*.json"))
    
    data = []
    for exp_file in experiment_files:
        with open(exp_file) as f:
            exp_data = json.load(f)
        data.append(exp_data)
    
    return data

def extract_metrics(data):
    metrics = {}
    param_names = list(data[0]['parameters'].keys())
    
    for exp in data:
        exp_id = exp['experiment_id']
        params = exp['parameters']
        
        avg_rankic = np.mean([r['metrics']['rank_ic'] for r in exp['results'].values()])
        avg_mse = np.mean([r['metrics']['mse'] for r in exp['results'].values()])
        
        if exp_id not in metrics:
            metrics[exp_id] = {'params': params, 'rank_ic': avg_rankic, 'mse': avg_mse}
    
    return metrics

results_data = load_results(output_dir)
metrics = extract_metrics(results_data)

print(f"Loaded {len(metrics)} experiments")

df_results = pd.DataFrame([
    {**m['params'], 'rank_ic': m['rank_ic'], 'mse': m['mse']} 
    for m in metrics.values()
])

print("\nResults summary:")
print(df_results.describe())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

param_names = [c for c in df_results.columns if c not in ['rank_ic', 'mse']]
for param in param_names:
    axes[0].scatter(df_results[param], df_results['rank_ic'], alpha=0.6, label=param)
axes[0].set_xlabel('Parameter Value')
axes[0].set_ylabel('Rank IC')
axes[0].set_title('Parameter Impact on Rank IC')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for param in param_names:
    axes[1].scatter(df_results[param], df_results['mse'], alpha=0.6, label=param)
axes[1].set_xlabel('Parameter Value')
axes[1].set_ylabel('MSE')
axes[1].set_title('Parameter Impact on MSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('03_sensitivity_analysis/data_parameters/results/sensitivity_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to: 03_sensitivity_analysis/data_parameters/results/sensitivity_overview.png")

## Sobol Sensitivity Indices (if Sobol method used)

In [ ]:
if METHOD == 'sobol':
    Y_rankic = np.array([m['rank_ic'] for m in metrics.values()])
    Y_mse = np.array([m['mse'] for m in metrics.values()])
    
    config = load_config()
    param_names = list(config['parameter_space'].keys())
    
    problem = {
        'num_vars': len(param_names),
        'names': param_names,
        'bounds': [
            [config['parameter_space'][p]['min'] - 0.5, 
             config['parameter_space'][p]['max'] + 0.5]
            for p in param_names
        ]
    }
    
    Si_rankic = sobol_analyze.analyze(problem, Y_rankic, calc_second_order=False, print_to_console=True)
    print("\n" + "="*60)
    print("MSE Sensitivity:")
    Si_mse = sobol_analyze.analyze(problem, Y_mse, calc_second_order=False, print_to_console=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].bar(param_names, Si_rankic['S1'], alpha=0.7, label='First-order')
    axes[0].bar(param_names, Si_rankic['ST'], alpha=0.5, label='Total-order')
    axes[0].set_ylabel('Sensitivity Index')
    axes[0].set_title('Rank IC Sensitivity')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].bar(param_names, Si_mse['S1'], alpha=0.7, label='First-order')
    axes[1].bar(param_names, Si_mse['ST'], alpha=0.5, label='Total-order')
    axes[1].set_ylabel('Sensitivity Index')
    axes[1].set_title('MSE Sensitivity')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('03_sensitivity_analysis/data_parameters/results/sobol_indices.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nSobol indices visualization saved to: 03_sensitivity_analysis/data_parameters/results/sobol_indices.png")
else:
    print("Sobol analysis only available for 'sobol' method")

## Summary

In [ ]:
print("="*80)
print("SENSITIVITY ANALYSIS COMPLETE")
print("="*80)
print(f"\nMethod: {METHOD}")
print(f"Experiments: {len(metrics)}")
print(f"\nBest Rank IC: {df_results['rank_ic'].max():.4f}")
best_idx = df_results['rank_ic'].idxmax()
print(f"Parameters: {df_results.loc[best_idx, param_names].to_dict()}")
print(f"\nWorst MSE: {df_results['mse'].min():.4f}")
best_mse_idx = df_results['mse'].idxmin()
print(f"Parameters: {df_results.loc[best_mse_idx, param_names].to_dict()}")
print("\nResults saved in: 03_sensitivity_analysis/data_parameters/results/")